 # Analysis Overview Flow PBMC Frequency Analysis
This notebook contains the following analyses:

## 1. NDMM subjects on D-VRd vs Healthy Donors

- Comparative analyses between participants in **D-VRd** of this study and **healthy donor** controls.
- Evaluation of differences across relevant clinical and/or molecular features.
- Statistical testing and visualization to assess group-level variation.

## 2. Paired Analyses Across Clinical Time Points (D-VRd)

- Longitudinal, paired analyses within **D-VRd** participants.
- Comparison of matched samples collected at different clinical time points.
- Within-subject statistical testing to evaluate temporal changes over the course of the study.
- Visualization of trajectories and paired differences.


In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(purrr)
  library(ggplot2)
  library(ggpubr)
  library(rstatix)
  library(forcats)
  library(tidyr)
  library(ggrepel)
  library(rlang)
  library(scales)
  
})
options(repr.plot.width = 11, repr.plot.height = 6)

## Load Metadata

In [2]:
### Load Metadata 
meta = readxl::read_xlsx('../data/flow/ndmm-rrmm-metadata-2025.xlsx',sheet = 1)
meta$subject.subjectGuid <- meta$Subject

### delete first 3 rows as 
### these are reference ranges 
### and descriptors for each column 
meta = meta[-c(1:3),]

### Refactor all 'BRI' subjects as healthy
meta[meta$Cohort %in% c('BR1','BR2'),]$Cohort <- 'Healthy'
table(meta$Cohort)


    FH1 Healthy 
    197      34 

## Load Compositional Data

In [3]:
### Load combined files 
mmFiles = fread('../data/flow/output/aggregated_flowdata.csv')

### Join metadata with data files
mmFiles <- dplyr::left_join(mmFiles, distinct(meta[,c('subject.subjectGuid','Sub_Cohort')]),
                      by='subject.subjectGuid')

mmFiles$label.visitDetails <- factor(mmFiles$label.visitDetails,
                                    levels=c('PreTx','PI2C','EI', 'Healthy'))

# Analysis
## DVRD NDMM Patients vs. Healthy

In [4]:
## Get cell pop names
uniqueCells = unique(mmFiles$aifi_label_l2)

## exclude healthy subjects
mm_patients = mmFiles[Sub_Cohort %in% c('Healthy','VRd_Dara')]

test_timePoint <- function(time='PreTx'){
    ## repeat for each tim
    preTX = mm_patients[label.visitDetails %in% c(time,'Healthy')]
    res <- lapply(uniqueCells,
                  function(x){
                      tmp=preTX[aifi_label_l2==x]
                      res = data.frame(
                          VRD = median(tmp[Sub_Cohort=='VRd_Dara']$pseudo_alc, na.rm=T),
                          Healthy =  median(tmp[Sub_Cohort=='Healthy']$pseudo_alc, na.rm=T),
                          panel= unique(tmp$panel),
                          Time=time,
                          Cell=x,
                          Pval=wilcox.test(tmp[Sub_Cohort=='VRd_Dara']$pseudo_alc,
                                           tmp[Sub_Cohort=='Healthy']$pseudo_alc)$p.value)
                      }
                  )
    res = rbindlist(res)
    res$FDR = p.adjust(res$Pval, method='fdr')
    res$Log2FC = log2(res$VRD) - log2(res$Healthy)
    return(res)    
}

healthy_comparisons <- rbindlist(lapply(c('PreTx','PI2C','EI'),
       function(x) test_timePoint(x)
       ))
head(healthy_comparisons)

Warning message in wilcox.test.default(tmp[Sub_Cohort == "VRd_Dara"]$pseudo_alc, :
“cannot compute exact p-value with ties”


VRD,Healthy,panel,Time,Cell,Pval,FDR,Log2FC
<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
4.882111,3.571510,PB1,PreTx,DN B cells,0.0824388982,0.267926419,0.4509711
6.399840,5.034079,PB1,PreTx,Effector B cells,0.6605449361,0.805039141,0.3463080
3.098910,7.098668,PB1,PreTx,Ig- memory,0.0044862645,0.021870539,-1.1957874
6.773719,8.017663,PB1,PreTx,IgA+ Memory,0.2214681929,0.541593785,-0.2432337
2.543144,5.869057,PB1,PreTx,IgG+ Memory,0.0002468126,0.002406423,-1.2065155
36.997732,51.299360,PB1,PreTx,IgM+ Naive,0.0453797860,0.176981165,-0.4715040


## Save files

In [5]:
write.csv(healthy_comparisons, file='../data/flow/output/dvrd_healthy.csv')